Split do guia inserido pelo parametro path:str

In [2]:
import io
from typing import List
from pypdf import PdfReader, PdfWriter

def split_pdf(path: str) -> List[bytes]:
    paginas_em_bytes = []
    
    reader = PdfReader(path)
    total_paginas = len(reader.pages)
    
    print(f"[Split] Cortando PDF com {total_paginas} páginas...")
    
    for idx in range(total_paginas):
        writer = PdfWriter()
        writer.add_page(reader.pages[idx])
        
        buffer = io.BytesIO()
        writer.write(buffer)
        

        paginas_em_bytes.append(buffer.getvalue())
        buffer.close()
        
    print(f"[Split] Concluído! {len(paginas_em_bytes)} páginas isoladas em memória.")
    return paginas_em_bytes

Vetorização das páginas do pdf

In [3]:
from typing import List
from google import genai
from google.genai import types
from dotenv import load_dotenv
from langchain_core.embeddings import Embeddings

load_dotenv()

client = genai.Client()


class GeminiEmbeddings(Embeddings):
    def __init__(self, client, model: str = "gemini-embedding-2-preview"):
        self.client = client
        self.model = model

    def _embed(self, contents, task_type: str) -> list[float]:
        resp = self.client.models.embed_content(
            model=self.model,
            contents=contents,
            config=types.EmbedContentConfig(task_type=task_type),
        )
        return resp.embeddings[0].values

    def embed_query(self, text: str) -> List[float]:
        return self._embed(text, task_type="RETRIEVAL_QUERY")

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return [self._embed(t, task_type="RETRIEVAL_DOCUMENT") for t in texts]

    def embed_pdf_bytes(self, page_bytes: bytes) -> List[float]:
        part = types.Part.from_bytes(data=page_bytes, mime_type="application/pdf")
        return self._embed(part, task_type="RETRIEVAL_DOCUMENT")


embeddings = GeminiEmbeddings(client)

Setup do vector store LangChain (Chroma persistente local)

In [4]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="guias_saude",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
    collection_metadata={"hnsw:space": "cosine"},
)

print(f"[Chroma] Coleção pronta. Itens atuais: {vector_store._collection.count()}")

[Chroma] Coleção pronta. Itens atuais: 276


In [ ]:
import os
from pypdf import PdfReader
from langchain_core.documents import Document


def extract_page_text(pdf_path: str, page_idx: int) -> str:
    return PdfReader(pdf_path).pages[page_idx].extract_text() or ""


def pipeline(pdf_path: str):
    paginas = split_pdf(pdf_path)
    source_name = os.path.basename(pdf_path)

    documents: list[Document] = []
    embeddings_list: list[list[float]] = []
    ids: list[str] = []

    for i, pagina_bytes in enumerate(paginas):
        num_pagina = i + 1
        print(f"[Pipeline] Vetorizando página {num_pagina}/{len(paginas)}...")

        vetor = embeddings.embed_pdf_bytes(pagina_bytes)
        page_text = extract_page_text(pdf_path, i)

        documents.append(
            Document(
                page_content=page_text,
                metadata={
                    "source": source_name,
                    "path": pdf_path,
                    "page": num_pagina,
                },
            )
        )
        embeddings_list.append(vetor)
        ids.append(f"{source_name}::page_{num_pagina}")

    vector_store._collection.upsert(
        ids=ids,
        documents=[d.page_content for d in documents],
        embeddings=embeddings_list,
        metadatas=[d.metadata for d in documents],
    )
    print(
        f"[Pipeline] {len(ids)} vetores salvos. "
        f"Total na coleção: {vector_store._collection.count()}"
    )

paths = [
    "guias/Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf",
    "guias/Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf",
]

if vector_store._collection.count() == 0:
    for guia in paths:
        pipeline(guia)
else:
    print("[Pipeline] Coleção já populada — ingestão pulada.")

### Exemplo de consulta

In [6]:
def query(pergunta: str, source: str | None = None, k: int = 3):
    filtro = {"source": source} if source else None
    return vector_store.similarity_search_with_score(pergunta, k=k, filter=filtro)


resultados = query(
    "Quais são os sintomas de hipoglicemia?"
)
for doc, score in resultados:
    print(
        f"  score={score:.4f}  {doc.metadata['source']}  "
        f"pág. {doc.metadata['page']}"
    )

  score=0.5800  Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf  pág. 98
  score=0.5813  Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf  pág. 8
  score=0.5943  Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf  pág. 62


In [7]:
resultados = query(
    "Classificação de risco do pé diabético", # "Diabetes gestacional"

    source="Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf",
)
for doc, score in resultados:
    print(
        f"  score={score:.4f}  {doc.metadata['source']}  "
        f"pág. {doc.metadata['page']}"
    )

# Exemplo extra: usando o vector store como Retriever do LangChain
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
docs = retriever.invoke("Classificação de risco do pé diabético")
print("\n[Retriever]")
for d in docs:
    print(f"  {d.metadata['source']}  pág. {d.metadata['page']}")

  score=0.5162  Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf  pág. 93
  score=0.5221  Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf  pág. 94
  score=0.5416  Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf  pág. 92

[Retriever]
  Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf  pág. 93
  Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf  pág. 94
  Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf  pág. 92


In [10]:
from IPython.display import Markdown
from agent import answer_question

resposta = answer_question(
    vector_store,
    client,
    "Estou com um paciente homem de 47 anos com estado hiperosmolar hiperglicêmico, como posso proceder?",
    k=5,
)

Markdown(resposta.as_markdown())

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [11]:
resposta = answer_question(
    vector_store,
    client,
    "Como proceder com uma paciente mulher de 31 anos, gestante, que está com infecção urinária?",
    k=5,
)

Markdown(resposta.as_markdown())

Para uma paciente gestante de 31 anos com infecção urinária (cistite aguda), as páginas anexadas fornecem as seguintes orientações:

1.  **Diagnóstico e Início do Tratamento:**
    *   A cistite aguda é a infecção do trato urinário baixo, com sintomas como disúria, polaciúria, incontinência urinária, dor suprapúbica e hematúria. Não ocorre febre, taquicardia, dor abdominal intensa, dor lombar, calafrios, náuseas ou vômitos (na presença desses sinais, suspeitar de pielonefrite e encaminhar para avaliação hospitalar via Vaga Zero) (pág. 64, 65).
    *   Não se deve aguardar o resultado dos exames (urocultura) para iniciar o tratamento quando o quadro é sugestivo (pág. 64).

2.  **Exames e Monitoramento:**
    *   A urocultura deve ser repetida duas semanas após o tratamento e mensalmente até o final da gestação (pág. 64).

3.  **Antibioticoterapia (Tratamento para Cistite Aguda):**
    A antibioticoterapia para cistite em gestantes, com duração de 7 dias, é a seguinte:

    *   **Nitrofurantoína (B):** 100mg de 6/6h (pág. 64).
    *   **Cefalexina (B):** 500mg de 6/6h (pág. 64).
    *   **Amoxicilina (B):** 500mg de 8/8h (pág. 64).
    *   **Sulfametoxazol + trimetoprima (C):** 400mg + 80mg (2 comprimidos de 12/12 horas).
        *   **Observações:** Usar em caso de *Escherichia coli* resistente; Evitar no primeiro trimestre e após a 32ª semana de gestação; Reforçar a importância da suplementação do ácido fólico 5mg/dia durante o tratamento (pág. 64).

4.  **Importante:**
    *   Em caso de EAS (Exame de Urina Simples) alterado em pacientes assintomáticas e com urocultura negativa, o tratamento para ITU não é indicado. Nesse caso, investigar outras causas para a alteração (pág. 64).
    *   Gestantes com infecção urinária de repetição (dois ou mais episódios na gestação, sintomáticos ou não; ou duas infecções nos últimos seis meses, ou três nos últimos 12 meses, antes da gestação) devem ter profilaxia considerada, com nitrofurantoína (100mg) ou cefalexina (500mg) via oral, uma dose à noite até duas semanas após o parto. Recomenda-se solicitar USG de vias urinárias (pág. 65).

**Fontes:**
- [Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf — pág. 65](guias/Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf#page=65)  (score=0.4890)
- [Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf — pág. 66](guias/Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf#page=66)  (score=0.5194)
- [Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf — pág. 64](guias/Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf#page=64)  (score=0.5232)
- [Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf — pág. 35](guias/Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf#page=35)  (score=0.5495)
- [Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf — pág. 95](guias/Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf#page=95)  (score=0.5508)

In [12]:
from IPython.display import Markdown, display

perguntas = [
    "Estou com um paciente homem de 47 anos com estado hiperosmolar hiperglicêmico, como posso proceder?",
    "Como proceder com uma paciente mulher de 31 anos, gestante, que está com infecção urinária?"
]

for i, pergunta in enumerate(perguntas, start=1):
    resposta = answer_question(vector_store, client, pergunta, k=10)
    display(Markdown(f"## Pergunta {i}\n\n**{pergunta}**\n\n---\n\n{resposta.as_markdown()}"))


## Pergunta 1

**Estou com um paciente homem de 47 anos com estado hiperosmolar hiperglicêmico, como posso proceder?**

---

Para um paciente homem de 47 anos com estado hiperosmolar hiperglicêmico (Síndrome Hiperglicêmica Hiperosmolar Não Cetótica - SHH), o procedimento a seguir, com base nas páginas fornecidas, é:

1.  **Correção da desidratação, dos distúrbios eletrolíticos e da hiperosmolaridade sérica** (pág. 101).
2.  **Correção da hiperglicemia** (pág. 101).
3.  **Identificação e tratamento dos fatores precipitantes** (pág. 101).
4.  **Em caso de suspeita de SHH, solicitar Vaga Zero, para confirmação do diagnóstico e internação** (pág. 101).

**Fontes:**
- [Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf — pág. 49](guias/Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf#page=49)  (score=0.5241)
- [Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf — pág. 100](guias/Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf#page=100)  (score=0.5360)
- [Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf — pág. 101](guias/Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf#page=101)  (score=0.5424)
- [Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf — pág. 8](guias/Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf#page=8)  (score=0.5506)
- [Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf — pág. 98](guias/Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf#page=98)  (score=0.5508)
- [Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf — pág. 102](guias/Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf#page=102)  (score=0.5548)
- [Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf — pág. 62](guias/Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf#page=62)  (score=0.5576)
- [Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf — pág. 47](guias/Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf#page=47)  (score=0.5626)
- [Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf — pág. 78](guias/Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf#page=78)  (score=0.5629)
- [Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf — pág. 64](guias/Livro_GuiaRapido-DiabetesMellitus_PDFDigital_20231113.pdf#page=64)  (score=0.5680)

## Pergunta 2

**Como proceder com uma paciente mulher de 31 anos, gestante, que está com infecção urinária?**

---

Para proceder com uma paciente mulher de 31 anos, gestante, com infecção urinária (cistite aguda), as seguintes orientações devem ser seguidas:

1.  **Avaliação dos Sintomas:**
    *   A cistite aguda é caracterizada por início agudo de sintomas urinários como disúria, polaciúria, incontinência urinária, dor suprapúbica e hematúria. (pág. 64)
    *   **Não ocorre febre, taquicardia, dor abdominal intensa, dor lombar, calafrios, náuseas ou vômitos na cistite aguda.** Se algum desses sinais estiver presente, deve-se suspeitar de pielonefrite, que é uma condição grave. (pág. 64)

2.  **Exames Laboratoriais:**
    *   O Exame de Urina Simples (EAS) pode evidenciar leucocitúria (acima de 10 leucócitos por campo) e hematúria. (pág. 64)
    *   A urocultura geralmente apresenta mais de 100 mil UFC/ml. (pág. 64)
    *   **Não se deve aguardar o resultado dos exames para iniciar o tratamento quando o quadro clínico é sugestivo.** (pág. 64)

3.  **Tratamento Antibiótico (Cistite Aguda):**
    *   A antibioticoterapia é indicada por 7 dias. As opções são:
        *   **Nitrofurantoína (B):** 100mg de 6/6h (pág. 64).
        *   **Cefalexina (B):** 500mg de 6/6h (pág. 64).
        *   **Amoxicilina (B):** 500mg de 8/8h (pág. 64).
        *   **Sulfametoxazol + trimetoprima (C):** 400mg + 80mg, 2 comprimidos de 12/12 horas. No entanto, este medicamento possui observações importantes:
            *   Deve ser usado em caso de *Escherichia coli* resistente. (pág. 64)
            *   **Evitar no primeiro trimestre e após a 32ª semana de gestação.** (pág. 64)
            *   Reforçar a importância da suplementação do ácido fólico 5mg/dia durante o tratamento. (pág. 64)

4.  **Atenção para Pielonefrite:**
    *   A pielonefrite aguda na gravidez é grave e caracteriza-se por sintomas sistêmicos como febre, taquicardia, calafrios, náusea, vômitos e dor lombar com sinal de Giordano positivo. (pág. 65)
    *   **IMPORTANTE: Em caso de suspeita de pielonefrite, encaminhar a gestante para avaliação hospitalar via Vaga Zero.** (pág. 65)

5.  **Acompanhamento Pós-Tratamento:**
    *   A urocultura deve ser repetida duas semanas após o tratamento e mensalmente até o final da gestação. (pág. 64)

**Fontes:**
- [Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf — pág. 65](guias/Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf#page=65)  (score=0.4890)
- [Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf — pág. 66](guias/Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf#page=66)  (score=0.5194)
- [Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf — pág. 64](guias/Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf#page=64)  (score=0.5232)
- [Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf — pág. 35](guias/Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf#page=35)  (score=0.5495)
- [Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf — pág. 95](guias/Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf#page=95)  (score=0.5508)
- [Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf — pág. 91](guias/Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf#page=91)  (score=0.5521)
- [Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf — pág. 85](guias/Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf#page=85)  (score=0.5532)
- [Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf — pág. 31](guias/Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf#page=31)  (score=0.5535)
- [Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf — pág. 98](guias/Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf#page=98)  (score=0.5550)
- [Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf — pág. 130](guias/Livro_GuiaRapido-PreNatal_PDFDigital_20250331.pdf#page=130)  (score=0.5593)